# Preprocesamiento y limpieza datasets

## 1. Datos no estructurados

En esta sección procesamos los dos archivos no estructurados del proyecto:
- `noticias_agricolas.jsonl`: Formato JSON Lines con noticias del sector agrícola.
- `reportes_plagas.txt`: Archivo de texto con registros de brotes de plagas.

El objetivo es transformar ambos archivos en DataFrames estructurados con columnas
de valor analítico.

**Formato:** JSON Lines (JSONL) — cada línea del archivo es un objeto JSON independiente.

**Campos identificados en exploración previa:**
| Campo             | Tipo        | Descripción                          |
|-------------------|-------------|--------------------------------------|
| `id_noticia`      | string      | Identificador único (NEWS####)       |
| `fecha`           | string      | Fecha de publicación (YYYY-MM-DD)    |
| `fuente`          | string      | Medio de comunicación                |
| `titular`         | string      | Título de la noticia                 |
| `contenido`       | string      | Cuerpo de la noticia                 |
| `impacto_cultivos`| list[str]   | Cultivos afectados (puede ser lista) |
| `severidad`       | string      | Nivel: baja / media / alta           |
| `paises_afectados`| list[str]   | Lista de países impactados           |
| `categoria`       | string      | clima / mercado / política / etc.    |

### 1.1. noticias_agricolas.jsonl

#### Celda 1 - Carga y exploración inicial

In [1]:
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# --- Carga línea a línea (formato JSONL) ---
registros = []
errores_carga = 0

with open('../data/raw/unstructured/noticias_agricolas.jsonl', 'r', encoding='utf-8') as f:
    for i, linea in enumerate(f):
        linea = linea.strip()
        if not linea:
            continue
        try:
            registros.append(json.loads(linea))
        except json.JSONDecodeError as e:
            errores_carga += 1
            print(f"  [WARN] Línea {i+1} no parseable: {e}")

df_noticias = pd.DataFrame(registros)

print(f"✅ Registros cargados : {len(df_noticias)}")
print(f"⚠️  Errores de carga  : {errores_carga}")
print(f"\n📋 Columnas detectadas ({len(df_noticias.columns)}):")
print(df_noticias.columns.tolist())
print(f"\n🔍 Primeras filas:")
df_noticias.head(3)

✅ Registros cargados : 60
⚠️  Errores de carga  : 0

📋 Columnas detectadas (9):
['fecha', 'fuente', 'titular', 'contenido', 'impacto_cultivos', 'severidad', 'paises_afectados', 'categoria', 'id_noticia']

🔍 Primeras filas:


,fecha,fuente,titular,contenido,impacto_cultivos,severidad,paises_afectados,categoria,id_noticia
0,2021-01-01,Bloomberg Commodities,Nueva política agrícola en Canadá busca mejora...,El gobierno de Canadá ha anunciado nuevas medi...,[todos],alta,[Australia],politica,NEWS5336
1,2021-01-11,Dairy Reporter,Ola de calor reduce rendimientos de Arroz,Eventos climáticos extremos están afectando la...,[Arroz],baja,"[India, Argentina, Rusia]",clima,NEWS8360
2,2021-01-19,World Grain,Sequía afecta proyecciones de cosecha en Alemania,Eventos climáticos extremos están afectando la...,[Maíz],alta,"[Argentina, México]",clima,NEWS6016


#### Celda 2 - Exploración tipos y nulos

In [2]:
print("=" * 55)
print("  INFORME DE EXPLORACIÓN — noticias_agricolas.jsonl")
print("=" * 55)

print(f"\n📐 Dimensiones: {df_noticias.shape[0]} filas × {df_noticias.shape[1]} columnas")

print("\n📊 Tipos de datos:")
print(df_noticias.dtypes)

print("\n🕳️  Valores nulos por columna:")
nulos = df_noticias.isnull().sum()
print(nulos[nulos >= 0].to_string())

print("\n📌 Valores únicos en columnas categóricas:")
for col in ['fuente', 'severidad', 'categoria']:
    print(f"  {col}: {df_noticias[col].nunique()} únicos → {df_noticias[col].unique().tolist()}")

print(f"\n📅 Rango temporal:")
print(f"  Desde : {df_noticias['fecha'].min()}")
print(f"  Hasta : {df_noticias['fecha'].max()}")

  INFORME DE EXPLORACIÓN — noticias_agricolas.jsonl

📐 Dimensiones: 60 filas × 9 columnas

📊 Tipos de datos:
fecha                  str
fuente                 str
titular                str
contenido              str
impacto_cultivos    object
severidad              str
paises_afectados    object
categoria              str
id_noticia             str
dtype: object

🕳️  Valores nulos por columna:
fecha               0
fuente              0
titular             0
contenido           0
impacto_cultivos    0
severidad           0
paises_afectados    0
categoria           0
id_noticia          0

📌 Valores únicos en columnas categóricas:
  fuente: 7 únicos → ['Bloomberg Commodities', 'Dairy Reporter', 'World Grain', 'Farm Journal', 'Reuters Agricultura', 'AgriCensus', 'Meat+Poultry']
  severidad: 3 únicos → ['alta', 'baja', 'media']
  categoria: 6 únicos → ['politica', 'clima', 'comercio', 'mercado', 'sanidad', 'tecnologia']

📅 Rango temporal:
  Desde : 2021-01-01
  Hasta : 2023-07-28


#### Celda 3 - Exploración de tipos y nulos

In [3]:
print("=" * 55)
print("  INFORME DE EXPLORACIÓN — noticias_agricolas.jsonl")
print("=" * 55)

print(f"\n📐 Dimensiones: {df_noticias.shape[0]} filas × {df_noticias.shape[1]} columnas")

print("\n📊 Tipos de datos:")
print(df_noticias.dtypes)

print("\n🕳️  Valores nulos por columna:")
nulos = df_noticias.isnull().sum()
print(nulos[nulos >= 0].to_string())

print("\n📌 Valores únicos en columnas categóricas:")
for col in ['fuente', 'severidad', 'categoria']:
    print(f"  {col}: {df_noticias[col].nunique()} únicos → {df_noticias[col].unique().tolist()}")

print(f"\n📅 Rango temporal:")
print(f"  Desde : {df_noticias['fecha'].min()}")
print(f"  Hasta : {df_noticias['fecha'].max()}")

  INFORME DE EXPLORACIÓN — noticias_agricolas.jsonl

📐 Dimensiones: 60 filas × 9 columnas

📊 Tipos de datos:
fecha                  str
fuente                 str
titular                str
contenido              str
impacto_cultivos    object
severidad              str
paises_afectados    object
categoria              str
id_noticia             str
dtype: object

🕳️  Valores nulos por columna:
fecha               0
fuente              0
titular             0
contenido           0
impacto_cultivos    0
severidad           0
paises_afectados    0
categoria           0
id_noticia          0

📌 Valores únicos en columnas categóricas:
  fuente: 7 únicos → ['Bloomberg Commodities', 'Dairy Reporter', 'World Grain', 'Farm Journal', 'Reuters Agricultura', 'AgriCensus', 'Meat+Poultry']
  severidad: 3 únicos → ['alta', 'baja', 'media']
  categoria: 6 únicos → ['politica', 'clima', 'comercio', 'mercado', 'sanidad', 'tecnologia']

📅 Rango temporal:
  Desde : 2021-01-01
  Hasta : 2023-07-28


**Decisiones de limpieza identificadas tras exploración:**

1. `fecha` es tipo `str` → convertir a `datetime64` y extraer `año` y `mes`
2. `impacto_cultivos` y `paises_afectados` son tipo `object` (listas Python) → 
   aplanar a string separado por `|` para compatibilidad CSV/Power BI
3. `severidad` y `categoria` pueden tener inconsistencias de capitalización → 
   normalizar a minúsculas
4. **Sin nulos detectados en ninguna columna** → no se requiere imputación
5. **60 registros** con rango 2021-01-01 a 2023-07-28 (~2.5 años de cobertura)
6. Verificar duplicados por `id_noticia` (no detectados a priori pero se comprueba 
   como buena práctica)

#### Celda 4 - Normalización y limpieza

In [4]:
df = df_noticias.copy()

# 1. Fecha a datetime
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df['año'] = df['fecha'].dt.year
df['mes']  = df['fecha'].dt.month

# 2. Aplanar listas a string delimitado por |
df['impacto_cultivos_str']  = df['impacto_cultivos'].apply(
    lambda x: '|'.join([str(v).strip().lower() for v in x]) if isinstance(x, list) else str(x).lower()
)
df['paises_afectados_str'] = df['paises_afectados'].apply(
    lambda x: '|'.join([str(v).strip() for v in x]) if isinstance(x, list) else str(x)
)
df['n_paises_afectados'] = df['paises_afectados'].apply(
    lambda x: len(x) if isinstance(x, list) else 1
)

# 3. Normalizar columnas de texto categórico
df['severidad']  = df['severidad'].str.strip().str.lower()
df['categoria']  = df['categoria'].str.strip().str.lower()
df['fuente']     = df['fuente'].str.strip()

# 4. Crear texto combinado para NLP (titular + contenido)
df['texto_nlp'] = df['titular'].fillna('') + '. ' + df['contenido'].fillna('')

# 5. Duplicados por id_noticia
duplicados = df.duplicated(subset='id_noticia', keep=False).sum()
print(f"🔁 Registros duplicados (id_noticia): {duplicados}")
df = df.drop_duplicates(subset='id_noticia', keep='first')

# 6. Nulos en fecha (conversión fallida)
fechas_nulas = df['fecha'].isnull().sum()
print(f"🕳️  Fechas no parseables: {fechas_nulas}")
if fechas_nulas > 0:
    df = df.dropna(subset=['fecha'])

print(f"\n✅ Dataset limpio: {df.shape[0]} registros × {df.shape[1]} columnas")
df[['id_noticia','fecha','fuente','severidad','categoria','impacto_cultivos_str','n_paises_afectados']].head(5)

🔁 Registros duplicados (id_noticia): 0
🕳️  Fechas no parseables: 0

✅ Dataset limpio: 60 registros × 15 columnas


,id_noticia,fecha,fuente,severidad,categoria,impacto_cultivos_str,n_paises_afectados
0,NEWS5336,2021-01-01,Bloomberg Commodities,alta,politica,todos,1
1,NEWS8360,2021-01-11,Dairy Reporter,baja,clima,arroz,3
2,NEWS6016,2021-01-19,World Grain,alta,clima,maíz,2
3,NEWS5711,2021-02-13,Farm Journal,media,comercio,todos,3
4,NEWS8036,2021-03-10,Reuters Agricultura,alta,mercado,carne_bovina,3


### Celda 5 - Análisis de Sentimiento
 
El enunciado plantea estudiar cómo las noticias afectan a los precios de mercado 
(Problema 4). Para ello necesitamos cuantificar si cada noticia transmite un 
mensaje positivo, negativo o neutro que pueda correlacionarse con variaciones de precio.

**Herramienta descartada: VADER (NLTK)**  
Se probó VADER como alternativa, pero al estar entrenado exclusivamente en inglés, 
clasifica texto en español de forma incorrecta: palabras negativas del dominio 
(`sequía`, `pérdidas`, `devastan`) no son reconocidas, resultando en 0 noticias 
negativas sobre un dataset que incluye eventos de pérdidas del 40%.

**Herramienta adoptada: Lexicón de dominio agrícola en español**  
Se implementa un "analizador" basado en léxico especificado manualmente con términos 
propios del sector agrícola en español. Esta solución es:
- Reproducible: sin dependencias externas ni modelos descargables
- Interpretable: las reglas son auditables y justificables
- Orientada al dominio: el lexicón refleja el vocabulario real del dataset
- Consistente: produce resultados estables y verificables

**Score resultante:** valor continuo en [-1.0, +1.0]  
**Etiqueta resultante:** `negativo` / `neutro` / `positivo`

**Observación sobre la distribución resultante:**  
El 75% de las noticias positivas presentan el mismo score máximo (0.75), 
lo que refleja una característica real del dataset: existe un subgrupo numeroso 
de noticias de políticas agrícolas con redacción prácticamente idéntica 
("busca mejorar sostenibilidad"), que activan siempre las mismas palabras positivas. Esto no invalida el análisis — es información útil sobre la 
homogeneidad del corpus — pero debe tenerse en cuenta al correlacionar 
sentimiento con precios en fases posteriores.

In [7]:
# Opción B: Lexicón de palabras clave en español orientado al dominio agrícola
# Solución robusta, interpretable y reproducible sin modelos externos

lexicon_negativo = [
    'sequía', 'inundaci', 'devastan', 'pérdidas', 'pérdida', 'afectando',
    'afecta', 'caída', 'bajando', 'baja', 'restricciones', 'restringe',
    'dañan', 'daño', 'crisis', 'riesgo', 'escasez', 'heladas', 'reducen',
    'reduce', 'alcanzan máximo', 'vulnerab'
]
lexicon_positivo = [
    'sostenible', 'sostenibilidad', 'mejorar', 'mejora', 'impulsa',
    'aumenta', 'subiendo', 'sube', 'exportaciones', 'crecimiento',
    'récord', 'innovación', 'tecnología', 'positiv', 'beneficio',
    'eficiencia', 'incremento', 'nuevo', 'avance'
]

def analizar_sentimiento_lexico(texto):
    if not isinstance(texto, str):
        return 0.0, 'neutro'
    texto_lower = texto.lower()
    score_neg = sum(1 for w in lexicon_negativo if w in texto_lower)
    score_pos = sum(1 for w in lexicon_positivo if w in texto_lower)
    score_neto = score_pos - score_neg
    if score_neto > 0:
        return round(min(score_neto * 0.15, 1.0), 4), 'positivo'
    elif score_neto < 0:
        return round(max(score_neto * 0.15, -1.0), 4), 'negativo'
    else:
        return 0.0, 'neutro'

resultados = df['texto_nlp'].apply(analizar_sentimiento_lexico)
df['sentiment_compound'] = resultados.apply(lambda x: x[0])
df['sentiment_label']    = resultados.apply(lambda x: x[1])

print("✅ Análisis de sentimiento (lexicón español) completado")
print(f"\n📊 Distribución de sentimiento:")
print(df['sentiment_label'].value_counts())
print(f"\n📈 Estadísticas del score compound:")
print(df['sentiment_compound'].describe().round(4))

✅ Análisis de sentimiento (lexicón español) completado

📊 Distribución de sentimiento:
sentiment_label
positivo    43
negativo    16
neutro       1
Name: count, dtype: int64

📈 Estadísticas del score compound:
count    60.0000
mean      0.3500
std       0.6284
min      -0.9000
25%      -0.1500
50%       0.7500
75%       0.7500
max       0.7500
Name: sentiment_compound, dtype: float64


#### Celda 6 - Validación y estadísticas output

In [8]:
print("=" * 55)
print("  VALIDACIÓN FINAL — noticias_processed")
print("=" * 55)

# Columnas del dataset final
cols_finales = [
    'id_noticia', 'fecha', 'año', 'mes', 'fuente',
    'titular', 'contenido', 'categoria', 'severidad',
    'impacto_cultivos_str', 'paises_afectados_str',
    'n_paises_afectados', 'sentiment_compound', 'sentiment_label'
]
df_final = df[cols_finales].copy()

print(f"\n📐 Dimensiones finales : {df_final.shape}")
print(f"🕳️  Nulos totales       : {df_final.isnull().sum().sum()}")
print(f"🔁 Duplicados          : {df_final.duplicated(subset='id_noticia').sum()}")

print(f"\n📅 Rango temporal: {df_final['fecha'].min().date()} → {df_final['fecha'].max().date()}")

print(f"\n📌 Categorías presentes:")
print(df_final['categoria'].value_counts().to_string())

print(f"\n🎭 Sentimiento por categoría:")
print(df_final.groupby('categoria')['sentiment_compound'].mean().round(3).sort_values())

df_final.head(5)

  VALIDACIÓN FINAL — noticias_processed

📐 Dimensiones finales : (60, 14)
🕳️  Nulos totales       : 0
🔁 Duplicados          : 0

📅 Rango temporal: 2021-01-01 → 2023-07-28

📌 Categorías presentes:
categoria
comercio      15
clima         11
tecnologia    10
mercado        9
sanidad        8
politica       7

🎭 Sentimiento por categoría:
categoria
clima        -0.805
mercado      -0.017
comercio      0.750
politica      0.750
sanidad       0.750
tecnologia    0.750
Name: sentiment_compound, dtype: float64


,id_noticia,fecha,año,mes,fuente,titular,contenido,categoria,severidad,impacto_cultivos_str,paises_afectados_str,n_paises_afectados,sentiment_compound,sentiment_label
0,NEWS5336,2021-01-01,2021,1,Bloomberg Commodities,Nueva política agrícola en Canadá busca mejora...,El gobierno de Canadá ha anunciado nuevas medi...,politica,alta,todos,Australia,1,0.75,positivo
1,NEWS8360,2021-01-11,2021,1,Dairy Reporter,Ola de calor reduce rendimientos de Arroz,Eventos climáticos extremos están afectando la...,clima,baja,arroz,India|Argentina|Rusia,3,-0.75,negativo
2,NEWS6016,2021-01-19,2021,1,World Grain,Sequía afecta proyecciones de cosecha en Alemania,Eventos climáticos extremos están afectando la...,clima,alta,maíz,Argentina|México,2,-0.75,negativo
3,NEWS5711,2021-02-13,2021,2,Farm Journal,Nueva política agrícola en Argentina busca mej...,El gobierno de Argentina ha anunciado nuevas m...,comercio,media,todos,India|México|Nueva Zelanda,3,0.75,positivo
4,NEWS8036,2021-03-10,2021,3,Reuters Agricultura,Demanda china impulsa mercado de carne_bovina,El mercado internacional de carne_bovina muest...,mercado,alta,carne_bovina,India|Argentina|Nueva Zelanda,3,-0.15,negativo


#### Celda 7 - Exportación

In [9]:
import os

output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)

archivo_salida = output_path + 'noticias_processed.csv'
df_final.to_csv(archivo_salida, index=False, encoding='utf-8-sig')

print(f"✅ Archivo exportado: {archivo_salida}")
print(f"   Registros : {len(df_final)}")
print(f"   Columnas  : {len(df_final.columns)}")
print(f"   Tamaño    : {os.path.getsize(archivo_salida) / 1024:.1f} KB")
print(f"\n📋 Columnas exportadas:")
for i, col in enumerate(df_final.columns, 1):
    print(f"   {i:02d}. {col}")

✅ Archivo exportado: ../data/processed/noticias_processed.csv
   Registros : 60
   Columnas  : 14
   Tamaño    : 20.8 KB

📋 Columnas exportadas:
   01. id_noticia
   02. fecha
   03. año
   04. mes
   05. fuente
   06. titular
   07. contenido
   08. categoria
   09. severidad
   10. impacto_cultivos_str
   11. paises_afectados_str
   12. n_paises_afectados
   13. sentiment_compound
   14. sentiment_label


**Conclusión Bloque A:**  

El dataset `noticias_processed.csv` contiene 60 registros limpios con 14 columnas,
incluyendo la feature de sentimiento generada por NLP. La categoría `clima` obtiene
el score medio más negativo (-0.805), lo que es coherente con el contenido real
de las noticias y valida el funcionamiento del lexicón. Este campo será clave
para correlacionar eventos noticiosos con variaciones de precios en Fase 3 (ML).

### 1.2. reportes_Plagas.txt